# Pipeline de Ensamblado y Anotación Genómica
**TFM — Trichoderma / Fungi genomics pipeline**

Cada bloque corresponde a una etapa del pipeline. Los binarios se llaman directamente por su ruta en el entorno conda correspondiente, sin necesidad de activar el entorno manualmente.

---
## 0. Configuración general
Definir rutas base, muestra activa, y binarios de cada entorno.

In [ ]:
import subprocess
import os

# ─── Rutas base ───────────────────────────────────────────────
BASE_DIR   = "/srv/TFM/PL-iteracion.05"   # cambia según iteración
MUESTRA    = "T16"                          # cambia según muestra
ENVS       = "/opt/miniconda3/envs"

# ─── Directorios del pipeline ─────────────────────────────────
DIR_RAW    = f"{BASE_DIR}/00.Archivos_principales/Secuencias_Illumina/{MUESTRA}"
DIR_TRIM   = f"{BASE_DIR}/01.Trimmomatic/{MUESTRA}"
DIR_ASM    = f"{BASE_DIR}/02.SPAdes_Assembly/{MUESTRA}"
DIR_PILON  = f"{BASE_DIR}/03.Pilon_Polished/{MUESTRA}"
DIR_QC     = f"{BASE_DIR}/X0.Reportes/{MUESTRA}"
DIR_STRUCT = f"{BASE_DIR}/05.Estructural_Funannotate/{MUESTRA}"
DIR_FUNC   = f"{BASE_DIR}/06.Funcional_Funannotate/{MUESTRA}"
DIR_SCRIPTS= f"{BASE_DIR}/0X.Scripts/default"

# ─── Binarios por entorno ─────────────────────────────────────
BIN = {
    "fastqc"      : f"{ENVS}/fastqc/bin/fastqc",
    "trimmomatic" : f"{ENVS}/trimmomatic/bin/trimmomatic",
    "spades"      : f"{ENVS}/spades/bin/spades.py",
    "pilon"       : f"{ENVS}/pilon/bin/pilon",
    "bwa"         : f"{ENVS}/pilon/bin/bwa",
    "samtools"    : f"{ENVS}/pilon/bin/samtools",
    "quast"       : f"{ENVS}/quast/bin/quast.py",
    "busco"       : f"{ENVS}/busco/bin/busco",
    "funannotate" : f"{ENVS}/funannotate/bin/funannotate",
    "seqkit"      : f"{ENVS}/seqkit/bin/seqkit",
}

# ─── Inputs de secuencias ─────────────────────────────────────
R1 = f"{DIR_RAW}/{MUESTRA}_1.fastq.gz"
R2 = f"{DIR_RAW}/{MUESTRA}_2.fastq.gz"

# Crear directorios si no existen
for d in [DIR_TRIM, DIR_ASM, DIR_PILON, DIR_QC, DIR_STRUCT, DIR_FUNC]:
    os.makedirs(d, exist_ok=True)

print(f"Muestra  : {MUESTRA}")
print(f"Base dir : {BASE_DIR}")
print(f"R1       : {R1}")
print(f"R2       : {R2}")

Muestra  : T16
Base dir : /srv/TFM/PL-iteracion.05
R1       : /srv/TFM/PL-iteracion.05/00.Archivos_principales/Secuencias_Illumina/T16/T16_1.fastq.gz
R2       : /srv/TFM/PL-iteracion.05/00.Archivos_principales/Secuencias_Illumina/T16/T16_2.fastq.gz


In [6]:
# ─── Helper para correr comandos ──────────────────────────────
def run(cmd, descripcion="", env_name="funannotate"):
    """Ejecuta un comando inyectando el PATH del entorno conda correspondiente."""
    if descripcion:
        print(f"▶ {descripcion}")
    print(f"  CMD: {' '.join(cmd)}\n")
    
    # Inyectar el PATH del entorno conda en el entorno de ejecución
    env = os.environ.copy()
    env_bin = f"/opt/miniconda3/envs/{env_name}/bin"
    env["PATH"] = env_bin + ":" + env["PATH"]
    
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode != 0:
        print(f"✗ Error (código {result.returncode})")
    else:
        print("✓ Completado")
    return result


---
## 1. FastQC — Control de calidad
**Entorno:** `fastqc`

In [2]:
DIR_FASTQC = f"{DIR_QC}/FastQC"
os.makedirs(DIR_FASTQC, exist_ok=True)

run([
    BIN["fastqc"],
    R1, R2,
    "--outdir", DIR_FASTQC,
    "--threads", "8"
], descripcion=f"FastQC sobre {MUESTRA}")

▶ FastQC sobre T16
  CMD: /opt/miniconda3/envs/fastqc/bin/fastqc /srv/TFM/PL-iteracion.05/00.Archivos_principales/Secuencias_Illumina/T16/T16_1.fastq.gz /srv/TFM/PL-iteracion.05/00.Archivos_principales/Secuencias_Illumina/T16/T16_2.fastq.gz --outdir /srv/TFM/PL-iteracion.05/X0.Reportes/T16/FastQC --threads 8

Can't exec "java": No such file or directory at /opt/miniconda3/envs/fastqc/bin/fastqc line 350.

✓ Completado


CompletedProcess(args=['/opt/miniconda3/envs/fastqc/bin/fastqc', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/Secuencias_Illumina/T16/T16_1.fastq.gz', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/Secuencias_Illumina/T16/T16_2.fastq.gz', '--outdir', '/srv/TFM/PL-iteracion.05/X0.Reportes/T16/FastQC', '--threads', '8'], returncode=0, stdout='', stderr='Can\'t exec "java": No such file or directory at /opt/miniconda3/envs/fastqc/bin/fastqc line 350.\n')

---
## 2. Trimmomatic — Trimming de lecturas
**Entorno:** `trimmomatic`

Ajusta `VERSION_TRIM` para distinguir distintos parámetros (ej. `v1_permisivo`, `v2_moderado`, `v3_estricto`).

In [ ]:
VERSION_TRIM = "v2_moderado"
ADAPTERS     = f"{ENVS}/trimmomatic/share/trimmomatic/adapters/TruSeq3-PE.fa"

DIR_TRIM_V   = f"{DIR_TRIM}/{MUESTRA}-trim-{VERSION_TRIM}"
os.makedirs(DIR_TRIM_V, exist_ok=True)

R1_PAIRED    = f"{DIR_TRIM_V}/{MUESTRA}_R1_paired.fq.gz"
R1_UNPAIRED  = f"{DIR_TRIM_V}/{MUESTRA}_R1_unpaired.fq.gz"
R2_PAIRED    = f"{DIR_TRIM_V}/{MUESTRA}_R2_paired.fq.gz"
R2_UNPAIRED  = f"{DIR_TRIM_V}/{MUESTRA}_R2_unpaired.fq.gz"

run([
    BIN["trimmomatic"], "PE",
    "-threads", "8",
    R1, R2,
    R1_PAIRED, R1_UNPAIRED,
    R2_PAIRED, R2_UNPAIRED,
    f"ILLUMINACLIP:{ADAPTERS}:2:30:10",
    "SLIDINGWINDOW:4:20",
    "MINLEN:50"
], descripcion=f"Trimmomatic {VERSION_TRIM} sobre {MUESTRA}")

---
## 3. SPAdes — Ensamblado
**Entorno:** `spades`

Ajusta `VERSION_SPADES` para distinguir configuraciones.

In [ ]:
VERSION_SPADES = "v1_rapido"

DIR_SPADES = f"{DIR_ASM}/{MUESTRA}-spades-{VERSION_TRIM}-spades{VERSION_SPADES}"
os.makedirs(DIR_SPADES, exist_ok=True)

run([
    "python", BIN["spades"],
    "-1", R1_PAIRED,
    "-2", R2_PAIRED,
    "--careful",
    "-t", "16",
    "-m", "64",
    "-o", DIR_SPADES
], descripcion=f"SPAdes {VERSION_SPADES} sobre {MUESTRA}")

CONTIGS = f"{DIR_SPADES}/scaffolds.fasta"

---
## 4. Filtrado con SeqKit
**Entorno:** `seqkit`

Filtra contigs por longitud mínima.

In [ ]:
MIN_LEN       = "500"
CONTIGS_FILT  = f"{DIR_SPADES}/scaffolds_filtered_{MIN_LEN}.fasta"

run([
    BIN["seqkit"], "seq",
    "-m", MIN_LEN,
    CONTIGS,
    "-o", CONTIGS_FILT
], descripcion=f"Filtrado contigs ≥{MIN_LEN}bp")

---
## 5. Pilon — Pulido del ensamblado
**Entorno:** `pilon`

Requiere mapear las lecturas al ensamblado con BWA + Samtools antes de correr Pilon.

In [ ]:
VERSION_PILON = "v1"
DIR_PILON_V   = f"{DIR_PILON}/{MUESTRA}-pilon-{VERSION_TRIM}-spades{VERSION_SPADES}-pilon{VERSION_PILON}"
os.makedirs(DIR_PILON_V, exist_ok=True)

BAM = f"{DIR_PILON_V}/reads_pe.bam"

# Indexar ensamblado
run([BIN["bwa"], "index", CONTIGS_FILT], descripcion="BWA index")

# Mapear lecturas
bwa_cmd = [
    BIN["bwa"], "mem", "-t", "16",
    CONTIGS_FILT, R1_PAIRED, R2_PAIRED
]
sort_cmd = [BIN["samtools"], "sort", "-o", BAM, "-"]

print("▶ BWA mem | samtools sort")
bwa_proc  = subprocess.Popen(bwa_cmd, stdout=subprocess.PIPE)
sort_proc = subprocess.Popen(sort_cmd, stdin=bwa_proc.stdout)
bwa_proc.stdout.close()
sort_proc.communicate()

# Indexar BAM
run([BIN["samtools"], "index", BAM], descripcion="Samtools index")

# Correr Pilon
run([
    BIN["pilon"],
    "--genome", CONTIGS_FILT,
    "--frags", BAM,
    "--outdir", DIR_PILON_V,
    "--output", f"{MUESTRA}_pilon",
    "--changes",
    "--threads", "16"
], descripcion=f"Pilon {VERSION_PILON}")

GENOME_POLISHED = f"{DIR_PILON_V}/{MUESTRA}_pilon.fasta"

---
## 6. QUAST — Evaluación del ensamblado
**Entorno:** `quast`

In [ ]:
DIR_QUAST = f"{DIR_QC}/QUAST"
os.makedirs(DIR_QUAST, exist_ok=True)

run([
    "python", BIN["quast"],
    GENOME_POLISHED,
    "-o", DIR_QUAST,
    "-t", "8"
], descripcion=f"QUAST sobre ensamblado pulido de {MUESTRA}")

---
## 7. BUSCO — Completitud del ensamblado
**Entorno:** `busco`

Ajusta `LINEAGE` según el organismo (ej. `fungi_odb10`, `ascomycota_odb10`).

In [ ]:
LINEAGE   = "ascomycota_odb10"
DIR_BUSCO = f"{DIR_QC}/BUSCO"
os.makedirs(DIR_BUSCO, exist_ok=True)

run([
    BIN["busco"],
    "-i", GENOME_POLISHED,
    "-o", f"{MUESTRA}_busco",
    "--out_path", DIR_BUSCO,
    "-l", LINEAGE,
    "-m", "genome",
    "-c", "8",
    "--force"
], descripcion=f"BUSCO ({LINEAGE}) sobre {MUESTRA}")

---
## 8. Funannotate — Anotación estructural
**Entorno:** `funannotate`

### Inicialización de la muestra

In [26]:
# ─── Entrada de esta sección ──────────────────────────────────
# Modificar estas dos variables antes de correr la sección completa

MUESTRA        = "T36"   # T16 / T22 / T36
GENOME_FILT    = f"{BASE_DIR}/03.Pulido_y_filtrado/{MUESTRA}/{MUESTRA}_filtered_3000.fasta"

# ─── Directorio de salida de Funannotate ──────────────────────
DIR_FUNANN = f"{BASE_DIR}/04.Funannotate/{MUESTRA}"
os.makedirs(DIR_FUNANN, exist_ok=True)

print(f"Muestra       : {MUESTRA}")
print(f"FASTA entrada : {GENOME_FILT}")
print(f"Directorio    : {DIR_FUNANN}")
print(f"Existe FASTA  : {os.path.exists(GENOME_FILT)}")

Muestra       : T36
FASTA entrada : /srv/TFM/PL-iteracion.05/03.Pulido_y_filtrado/T36/T36_filtered_3000.fasta
Directorio    : /srv/TFM/PL-iteracion.05/04.Funannotate/T36
Existe FASTA  : True


### Funannotate Clean

In [27]:
# Elimina contigs cortos y redundantes (duplicados por cobertura)
# Input  : FASTA filtrado por longitud (≥3000bp ya aplicado)
# Output : FASTA limpio sin contigs redundantes

GENOME_CLEAN = f"{DIR_FUNANN}/{MUESTRA}_clean.fasta"

run([
    BIN["funannotate"], "clean",
    "-i", GENOME_FILT,
    "-o", GENOME_CLEAN,
    "--minlen", "500",        # longitud mínima de contig a conservar (ya filtraste a 3000, puedes bajar esto)
    "--pident", "95",         # % identidad para considerar un contig redundante
    "--cov",    "95",         # % cobertura para considerar un contig redundante
    # "--exhaustive",         # búsqueda más exhaustiva de redundancias, más lento
], descripcion=f"Funannotate clean — {MUESTRA}", env_name="funannotate")


▶ Funannotate clean — T36
  CMD: /opt/miniconda3/envs/funannotate/bin/funannotate clean -i /srv/TFM/PL-iteracion.05/03.Pulido_y_filtrado/T36/T36_filtered_3000.fasta -o /srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_clean.fasta --minlen 500 --pident 95 --cov 95

-----------------------------------------------
204 input contigs, 204 larger than 500 bp, N50 is 916,386 bp
Checking duplication of 189 contigs shorter than N50
-----------------------------------------------
-----------------------------------------------
204 input contigs; 204 larger than 500 bp; 0 duplicated; 204 written to file

/opt/miniconda3/envs/funannotate/lib/python3.11/site-packages/funannotate/funannotate.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution
minimap2 version=2.31

CompletedProcess(args=['/opt/miniconda3/envs/funannotate/bin/funannotate', 'clean', '-i', '/srv/TFM/PL-iteracion.05/03.Pulido_y_filtrado/T36/T36_filtered_3000.fasta', '-o', '/srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_clean.fasta', '--minlen', '500', '--pident', '95', '--cov', '95'], returncode=0, stdout='-----------------------------------------------\n204 input contigs, 204 larger than 500 bp, N50 is 916,386 bp\nChecking duplication of 189 contigs shorter than N50\n-----------------------------------------------\n-----------------------------------------------\n204 input contigs; 204 larger than 500 bp; 0 duplicated; 204 written to file\n', stderr='/opt/miniconda3/envs/funannotate/lib/python3.11/site-packages/funannotate/funannotate.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.\n 

### Funannotate Sort

In [28]:
# Renombra headers de contigs a formato simple (scaffold_1, scaffold_2...)
# Funannotate es sensible a headers largos o con caracteres especiales
# Input  : FASTA limpio
# Output : FASTA con headers normalizados

GENOME_SORTED = f"{DIR_FUNANN}/{MUESTRA}_sorted.fasta"

run([
    BIN["funannotate"], "sort",
    "-i", GENOME_CLEAN,
    "-o", GENOME_SORTED,
    "-b", f"scaffold",        # prefijo para los headers (ej. scaffold_1, scaffold_2...)
    "--minlen", "0",      # filtro adicional de longitud mínima si no se hizo en clean
], descripcion=f"Funannotate sort — {MUESTRA}", env_name="funannotate")

▶ Funannotate sort — T36
  CMD: /opt/miniconda3/envs/funannotate/bin/funannotate sort -i /srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_clean.fasta -o /srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_sorted.fasta -b scaffold --minlen 0

204 contigs records loaded
Sorting and renaming contig headers
204 contigs saved to file

/opt/miniconda3/envs/funannotate/lib/python3.11/site-packages/funannotate/funannotate.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution

✓ Completado


CompletedProcess(args=['/opt/miniconda3/envs/funannotate/bin/funannotate', 'sort', '-i', '/srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_clean.fasta', '-o', '/srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_sorted.fasta', '-b', 'scaffold', '--minlen', '0'], returncode=0, stdout='204 contigs records loaded\nSorting and renaming contig headers\n204 contigs saved to file\n', stderr='/opt/miniconda3/envs/funannotate/lib/python3.11/site-packages/funannotate/funannotate.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.\n  from pkg_resources import get_distribution\n')

### Funannotate Mask

In [29]:
# Soft-masking de regiones repetitivas
# Input  : FASTA con headers normalizados
# Output : FASTA con repeticiones en minúsculas (soft-masked)

GENOME_MASKED = f"{DIR_FUNANN}/{MUESTRA}_masked.fasta"

run([
    BIN["funannotate"], "mask",
    "-i", GENOME_SORTED,
    "-o", GENOME_MASKED,
    "--cpus", "16",
    # "--method", "repeatmasker",  # usar RepeatMasker si está instalado (más completo)
    # "--method", "tantan",        # usar Tantan (por defecto, más rápido, menos completo)
    # "--repeatlib", "/ruta/custom.lib",  # librería custom de repetitivos si tienes una de Trichoderma
], descripcion=f"Funannotate mask — {MUESTRA}")

▶ Funannotate mask — T36
  CMD: /opt/miniconda3/envs/funannotate/bin/funannotate mask -i /srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_sorted.fasta -o /srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_masked.fasta --cpus 16

-------------------------------------------------------
-------------------------------------------------------

/opt/miniconda3/envs/funannotate/lib/python3.11/site-packages/funannotate/funannotate.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution
[May 30 08:44 PM]: OS: Ubuntu 24.04, 16 cores, ~ 32 GB RAM. Python: 3.11.15
[May 30 08:44 PM]: Running funanotate v1.8.17
[May 30 08:44 PM]: Soft-masking simple repeats with tantan
[May 30 08:45 PM]: Repeat soft-masking finished: 
Masked genome: /srv/TFM/PL-iteracion.05/04.Funannot

CompletedProcess(args=['/opt/miniconda3/envs/funannotate/bin/funannotate', 'mask', '-i', '/srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_sorted.fasta', '-o', '/srv/TFM/PL-iteracion.05/04.Funannotate/T36/T36_masked.fasta', '--cpus', '16'], returncode=0, stdout='-------------------------------------------------------\n-------------------------------------------------------\n', stderr='/opt/miniconda3/envs/funannotate/lib/python3.11/site-packages/funannotate/funannotate.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.\n  from pkg_resources import get_distribution\n[May 30 08:44 PM]: OS: Ubuntu 24.04, 16 cores, ~ 32 GB RAM. Python: 3.11.15\n[May 30 08:44 PM]: Running funanotate v1.8.17\n[May 30 08:44 PM]: Soft-masking simple repeats with tantan\n[May 30 08:45 PM]: Repeat soft-masking finished

### Funannotate Train

In [ ]:
# ─── Funannotate train — T16 (T. asperellum) ─────────────────
# Usa los 3 pares de RNA-seq de T. asperellum
# Trinity corre internamente — puede tardar varias horas
# Recomendado correr en nohup o tmux

SRA_ASPERELLUM = f"{BASE_DIR}/00.Archivos_principales/SRA-seq/T_asperellum"

run([
    BIN["funannotate"], "train",
    "-i", GENOME_MASKED,
    "-o", DIR_FUNANN,
    "--left",
        f"{SRA_ASPERELLUM}/SRR12495788_1.fastq.gz",
        f"{SRA_ASPERELLUM}/SRR12495790_1.fastq.gz",
        f"{SRA_ASPERELLUM}/SRR34502155_1.fastq.gz",
    "--right",
        f"{SRA_ASPERELLUM}/SRR12495788_2.fastq.gz",
        f"{SRA_ASPERELLUM}/SRR12495790_2.fastq.gz",
        f"{SRA_ASPERELLUM}/SRR34502155_2.fastq.gz",
    "--species", "Trichoderma asperellum",
    "--strain", MUESTRA,
    "--cpus", "16",
    "--memory", "26G",      # deja ~4GB libres de tus 32GB
    "--no_normalize_reads", # omite normalización, los reads ya son de buena calidad
    "--PASAHOME", "/opt/miniconda3/envs/funannotate/bin",
    "--TRINITYHOME", "/opt/miniconda3/envs/funannotate/bin",
    #"--SEQCLEAN", "/opt/miniconda3/envs/funannotate/bin/",
    # "--stranded", "RF",   # descomenta si sabes que la librería es stranded RF
    # "--stranded", "FR",   # o FR según el protocolo de secuenciación
], descripcion=f"Funannotate train — {MUESTRA}", env_name="funannotate")

▶ Funannotate train — T16
  CMD: /opt/miniconda3/envs/funannotate/bin/funannotate train -i /srv/TFM/PL-iteracion.05/04.Funannotate/T16/T16_masked.fasta -o /srv/TFM/PL-iteracion.05/04.Funannotate/T16 --left /srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495788_1.fastq.gz /srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495790_1.fastq.gz /srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR34502155_1.fastq.gz --right /srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495788_2.fastq.gz /srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495790_2.fastq.gz /srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR34502155_2.fastq.gz --species Trichoderma asperellum --strain T16 --cpus 16 --memory 26G --no_normalize_reads --PASAHOME /opt/miniconda3/envs/funannotate/bin --TRINITYHOME /opt/miniconda3/envs/funannotate/bin --SEQCLEAN /opt/miniconda3/envs/funanno

CompletedProcess(args=['/opt/miniconda3/envs/funannotate/bin/funannotate', 'train', '-i', '/srv/TFM/PL-iteracion.05/04.Funannotate/T16/T16_masked.fasta', '-o', '/srv/TFM/PL-iteracion.05/04.Funannotate/T16', '--left', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495788_1.fastq.gz', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495790_1.fastq.gz', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR34502155_1.fastq.gz', '--right', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495788_2.fastq.gz', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR12495790_2.fastq.gz', '/srv/TFM/PL-iteracion.05/00.Archivos_principales/SRA-seq/T_asperellum/SRR34502155_2.fastq.gz', '--species', 'Trichoderma asperellum', '--strain', 'T16', '--cpus', '16', '--memory', '26G', '--no_normalize_reads', '--PASAHOME', '/opt/miniconda3/envs/funannotate/bin', '--TRINITYHOME', '/opt/mi

### Funannotate Predict (estructural)

In [ ]:
# Predicción estructural de genes
# Si corriste train, funannotate detecta automáticamente el output en DIR_FUNANN
# Si no corriste train, añade --protein_evidence

SPECIES = "Trichoderma reesei"   # usa la especie conocida, o la más cercana en Augustus
STRAIN  = MUESTRA                 # para cepas desconocidas, el ID de cepa es suficiente

# Comprueba especies disponibles en Augustus con:
# augustus --species=help 2>&1 | grep -i trichoderma

run([
    BIN["funannotate"], "predict",
    "-i", GENOME_MASKED,
    "-o", DIR_FUNANN,
    "-s", SPECIES,
    "--strain",    STRAIN,
    "--busco_db", "hypocreales_odb12",
    "--busco_seed_species","fusarium_graminearum",
    "--organism", "fungal",
    "--cpus",      "16",
    # "--protein_evidence", "/ruta/trichoderma_proteins.fasta",  # recomendado si no hay RNA-seq
    # "--transcript_evidence", "/ruta/transcripts.fasta",        # transcritos de referencia opcionales
    # "--busco_seed_species", "trichoderma_reesei",              # si no se reconoce la especie en Augustus
    # "--organism", "fungal",                                    # optimizaciones específicas para hongos
    # "--repeats2evm",                                           # pasa info de repetitivos a EvidenceModeler
], descripcion=f"Funannotate predict — {MUESTRA}")

---
## 9. Funannotate — Anotación funcional
**Entorno:** `funannotate`

In [ ]:
DIR_FUNC_V = f"{DIR_FUNC}/{MUESTRA}"
os.makedirs(DIR_FUNC_V, exist_ok=True)

run([
    BIN["funannotate"], "annotate",
    "-i", DIR_FUNANN,
    "--cpus", "16",
    "-o", DIR_FUNC_V
], descripcion="Funannotate annotate")

---
*Fin del pipeline. Revisa los reportes en `X0.Reportes/`.*